In [1]:
# === ÉTAPE 1: INSTALLATION ET IMPORTS ===

# Installation des dépendances (décommenter si nécessaire)
# !pip install -q torch torchaudio librosa scikit-learn matplotlib pandas tqdm

import os
import json
import time
from collections import defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import librosa
import soundfile as sf

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier

print("✅ Toutes les imports sont réussis!")
print(f"🐍 Version PyTorch: {torch.__version__}")

✅ Toutes les imports sont réussis!
🐍 Version PyTorch: 2.9.0


In [2]:
# === ÉTAPE 2: CONFIGURATION ===

# Configuration des chemins (à adapter selon ton setup)
AUDIO_DIR = "/Users/lafiloche31200/Projects/projet_donnee/audio"
JSON_PATH = "/Users/lafiloche31200/Projects/projet_donnee/train_val_annotation/train_val_videodatainfo.json"

# Configuration du device (GPU/MPS/CPU)
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

# Configuration modèle
NUM_CLASSES = 20
RANDOM_STATE = 42
BATCH_SIZE = 16

print("⚙️ Configuration:")
print(f"  - Device: {DEVICE}")
print(f"  - Classes: {NUM_CLASSES}")
print(f"  - Batch Size: {BATCH_SIZE}")

# Désactivation des warnings macOS pour multiprocessing
import os
os.environ['OBJC_DISABLE_INITIALIZE_FORK_SAFETY'] = 'YES'

⚙️ Configuration:
  - Device: mps
  - Classes: 20
  - Batch Size: 16


In [3]:
# === ÉTAPE 3: CHARGEMENT DES MÉTADONNÉES ===

print("📊 Chargement des métadonnées...")

# Chargement du fichier JSON
with open(JSON_PATH, "r") as f:
    meta = json.load(f)

# Mapping des catégories
categories = {
    0: "music", 1: "people", 2: "gaming", 3: "sports/actions", 4: "news/events/politics",
    5: "education", 6: "tv shows", 7: "movie/comedy", 8: "animation", 9: "vehicles/autos",
    10: "howto", 11: "travel", 12: "science/technology", 13: "animals/pets",
    14: "kids/family", 15: "documentary", 16: "food/drink", 17: "cooking",
    18: "beauty/fashion", 19: "advertisement"
}

# Création du DataFrame des vidéos
videos = meta["videos"]
df_videos = pd.DataFrame(videos)
df_videos["category_name"] = df_videos["category"].map(categories)

# Construction de la liste des fichiers audio avec labels
audio_files, labels = [], []
for fname in os.listdir(AUDIO_DIR):
    if not fname.lower().endswith(".wav"):
        continue
    vid_id = os.path.splitext(fname)[0]
    row = df_videos[df_videos["video_id"] == vid_id]
    if not row.empty:
        audio_files.append(os.path.join(AUDIO_DIR, fname))
        labels.append(int(row.iloc[0]["category"]))

df_audio = pd.DataFrame({"file_path": audio_files, "label": labels})

print(f"✅ {len(df_audio)} fichiers audio chargés")
print(f"📊 Distribution des classes:")
print(df_audio["label"].value_counts().sort_index())

📊 Chargement des métadonnées...
✅ 6176 fichiers audio chargés
📊 Distribution des classes:
label
0     406
1     155
2     285
3     733
4     314
5     171
6     175
7     594
8     189
9     520
10    327
11    175
12    232
13    388
14    345
15    104
16    430
17    218
18    324
19     91
Name: count, dtype: int64


In [4]:
# === ÉTAPE 4: EXTRACTION DES FEATURES AUDIO ===

print("🎵 Extraction des features audio...")

def extract_mfcc_np(file_path, sr=16000, n_mfcc=40):
    """Extraction MFCC (pour modèles simples)"""
    try:
        y, _ = librosa.load(file_path, sr=sr, mono=True)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        return np.mean(mfcc.T, axis=0)  # shape (n_mfcc,)
    except Exception as e:
        print(f"❌ Erreur MFCC {file_path}: {e}")
        return np.zeros(n_mfcc)

def extract_mel_np(file_path, sr=16000, n_mels=128, n_fft=1024, hop_length=512):
    """Extraction Mel-spectrogramme (pour CNN/CRNN)"""
    try:
        y, _ = librosa.load(file_path, sr=sr, mono=True)
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels,
                                             n_fft=n_fft, hop_length=hop_length)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        return mel_db.astype(np.float32)  # shape (n_mels, T)
    except Exception as e:
        print(f"❌ Erreur Mel {file_path}: {e}")
        return np.zeros((n_mels, 100))

def pad_truncate_spec(spec, max_len=400):
    """Padding/truncation des spectrogrammes"""
    n_mels, T = spec.shape
    if T >= max_len:
        return spec[:, :max_len]
    out = np.zeros((n_mels, max_len), dtype=spec.dtype)
    out[:, :T] = spec
    return out

print("✅ Fonctions d'extraction définies")

🎵 Extraction des features audio...
✅ Fonctions d'extraction définies


In [5]:
# === ÉTAPE 5: BASELINE RANDOM FOREST ===

print("🌲 Entraînement Random Forest (baseline)...")

# Extraction des MFCCs
X_mfcc = []
for fp in tqdm(df_audio["file_path"].values, desc="Extraction MFCCs"):
    X_mfcc.append(extract_mfcc_np(fp, n_mfcc=40))
X_mfcc = np.vstack(X_mfcc)
y = df_audio["label"].values

# Split train/test
X_tr, X_te, y_tr, y_te = train_test_split(
    X_mfcc, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Entraînement Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_tr, y_tr)
y_pred_rf = rf.predict(X_te)

# Résultats
rf_accuracy = accuracy_score(y_te, y_pred_rf)
rf_f1 = f1_score(y_te, y_pred_rf, average='macro', zero_division=0)

print(f"✅ Random Forest Baseline:")
print(f"   - Accuracy: {rf_accuracy:.4f}")
print(f"   - F1-Score: {rf_f1:.4f}")

🌲 Entraînement Random Forest (baseline)...


Extraction MFCCs: 100%|██████████| 6176/6176 [00:42<00:00, 144.53it/s]


✅ Random Forest Baseline:
   - Accuracy: 0.3115
   - F1-Score: 0.2392


In [6]:
# === ÉTAPE 6: DATASETS ET DATALOADERS ===

print("📦 Création des Datasets PyTorch...")

class SpecDataset(Dataset):
    """Dataset pour spectrogrammes (CNN/CRNN)"""
    def __init__(self, df, max_len=400, n_mels=128):
        self.df = df.reset_index(drop=True)
        self.max_len = max_len
        self.n_mels = n_mels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        fp = self.df.loc[idx, "file_path"]
        label = int(self.df.loc[idx, "label"])
        
        spec = extract_mel_np(fp, n_mels=self.n_mels)
        spec = pad_truncate_spec(spec, max_len=self.max_len)
        
        # Normalisation
        spec = (spec - np.mean(spec)) / (np.std(spec) + 1e-9)
        spec_tensor = torch.tensor(spec).unsqueeze(0)  # (1, n_mels, T)
        
        return spec_tensor.float(), torch.tensor(label).long()

# Split des données
df_train, df_test = train_test_split(
    df_audio, test_size=0.2, random_state=RANDOM_STATE, stratify=df_audio["label"]
)
df_train, df_val = train_test_split(
    df_train, test_size=0.1, random_state=RANDOM_STATE, stratify=df_train["label"]
)

# Création des datasets
train_ds = SpecDataset(df_train)
val_ds = SpecDataset(df_val)
test_ds = SpecDataset(df_test)

# DataLoaders (num_workers=0 pour éviter les problèmes macOS)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"✅ Datasets créés:")
print(f"   - Train: {len(train_ds)} samples")
print(f"   - Val: {len(val_ds)} samples")
print(f"   - Test: {len(test_ds)} samples")

📦 Création des Datasets PyTorch...
✅ Datasets créés:
   - Train: 4446 samples
   - Val: 494 samples
   - Test: 1236 samples


In [7]:
# === ÉTAPE 7: DÉFINITION DES MODÈLES ===

print("🧠 Définition des architectures de modèles...")

# Modèle 1: CNN Simple
class AudioCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1))
        )
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

# Modèle 2: CRNN (CNN + RNN)
class CRNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, n_mels=128, gru_hidden=128, gru_layers=1):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d((2,2)),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d((2,2)),
        )
        self.gru = nn.GRU(
            input_size=(n_mels//4)*32, 
            hidden_size=gru_hidden, 
            num_layers=gru_layers, 
            batch_first=True, 
            bidirectional=True
        )
        self.fc = nn.Sequential(
            nn.Linear(gru_hidden*2, 256), 
            nn.ReLU(), 
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        b = x.size(0)
        x = self.cnn(x)
        c, f, t = x.size(1), x.size(2), x.size(3)
        x = x.permute(0, 3, 1, 2)
        x = x.contiguous().view(b, t, c*f)
        out, _ = self.gru(x)
        out = out.mean(dim=1)
        return self.fc(out)

# Modèle 3: Deep CNN
class DeepAudioCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            # Bloc 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.25),
            
            # Bloc 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.25),
            
            # Bloc 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.25),
            
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

print("✅ Modèles définis: CNN, CRNN, DeepCNN")

🧠 Définition des architectures de modèles...
✅ Modèles définis: CNN, CRNN, DeepCNN


In [8]:
# === ÉTAPE 8: FONCTIONS D'ENTRAÎNEMENT ===

print("⚙️ Définition des fonctions d'entraînement...")

def train_one_epoch(model, loader, optimizer, criterion, device):
    """Entraînement pour une epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * X.size(0)
        preds = out.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += X.size(0)
    
    return running_loss/total, correct/total

def evaluate(model, loader, device):
    """Évaluation du modèle"""
    model.eval()
    preds_all = []
    labels_all = []
    
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            preds = out.argmax(dim=1)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(y.cpu().numpy())
    
    acc = accuracy_score(labels_all, preds_all)
    f1 = f1_score(labels_all, preds_all, average='macro', zero_division=0)
    return acc, f1, labels_all, preds_all

def train_model(model, train_loader, val_loader, model_name, epochs=5, lr=1e-3):
    """Fonction complète d'entraînement"""
    print(f"\n🚀 Entraînement {model_name}...")
    
    model = model.to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    best_val_acc = 0
    history = {'train_loss': [], 'train_acc': [], 'val_acc': []}
    
    for epoch in range(epochs):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_acc, val_f1, _, _ = evaluate(model, val_loader, DEVICE)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
        
        print(f"[{model_name}] Epoch {epoch+1}/{epochs} | "
              f"Loss={train_loss:.4f} | Train Acc={train_acc:.4f} | "
              f"Val Acc={val_acc:.4f} | Time={(time.time()-t0):.1f}s")
    
    # Test final
    test_acc, test_f1, _, _ = evaluate(model, test_loader, DEVICE)
    print(f"✅ {model_name} - Test: Acc={test_acc:.4f}, F1={test_f1:.4f}")
    
    return test_acc, test_f1, history

print("✅ Fonctions d'entraînement définies")

⚙️ Définition des fonctions d'entraînement...
✅ Fonctions d'entraînement définies


In [9]:
# === ÉTAPE 9: ENTRAÎNEMENT DES MODÈLES DE BASE ===

print("🔥 ENTRAÎNEMENT DES MODÈLES DE BASE")
print("=" * 60)

# Dictionnaire pour stocker les résultats
results = {}

# 1. CNN Simple
print("\n🎵 1. Entraînement CNN Simple...")
model_cnn = AudioCNN(num_classes=NUM_CLASSES)
cnn_acc, cnn_f1, cnn_history = train_model(
    model_cnn, train_loader, val_loader, "CNN", epochs=5, lr=1e-3
)
results['CNN'] = (cnn_acc, cnn_f1)

# 2. CRNN
print("\n🎵 2. Entraînement CRNN...")
model_crnn = CRNN(num_classes=NUM_CLASSES)
crnn_acc, crnn_f1, crnn_history = train_model(
    model_crnn, train_loader, val_loader, "CRNN", epochs=5, lr=1e-3
)
results['CRNN'] = (crnn_acc, crnn_f1)

# 3. Deep CNN
print("\n🎵 3. Entraînement Deep CNN...")
model_deep = DeepAudioCNN(num_classes=NUM_CLASSES)
deep_acc, deep_f1, deep_history = train_model(
    model_deep, train_loader, val_loader, "Deep CNN", epochs=5, lr=1e-3
)
results['Deep CNN'] = (deep_acc, deep_f1)

# Ajout du Random Forest aux résultats
results['Random Forest'] = (rf_accuracy, rf_f1)

print("\n✅ TOUS LES MODÈLES DE BASE ONT ÉTÉ ENTRAÎNÉS")

🔥 ENTRAÎNEMENT DES MODÈLES DE BASE

🎵 1. Entraînement CNN Simple...

🚀 Entraînement CNN...
[CNN] Epoch 1/5 | Loss=2.8246 | Train Acc=0.1352 | Val Acc=0.1215 | Time=35.4s
[CNN] Epoch 2/5 | Loss=2.7657 | Train Acc=0.1547 | Val Acc=0.1559 | Time=30.0s
[CNN] Epoch 3/5 | Loss=2.7497 | Train Acc=0.1633 | Val Acc=0.1457 | Time=29.7s
[CNN] Epoch 4/5 | Loss=2.7355 | Train Acc=0.1590 | Val Acc=0.1579 | Time=34.1s
[CNN] Epoch 5/5 | Loss=2.7133 | Train Acc=0.1709 | Val Acc=0.1478 | Time=33.0s
✅ CNN - Test: Acc=0.1553, F1=0.0756

🎵 2. Entraînement CRNN...

🚀 Entraînement CRNN...
[CRNN] Epoch 1/5 | Loss=2.8621 | Train Acc=0.1210 | Val Acc=0.1559 | Time=46.3s
[CRNN] Epoch 2/5 | Loss=2.7539 | Train Acc=0.1516 | Val Acc=0.1660 | Time=44.9s
[CRNN] Epoch 3/5 | Loss=2.6545 | Train Acc=0.1811 | Val Acc=0.1640 | Time=45.5s
[CRNN] Epoch 4/5 | Loss=2.5675 | Train Acc=0.2081 | Val Acc=0.1943 | Time=51.0s
[CRNN] Epoch 5/5 | Loss=2.4709 | Train Acc=0.2438 | Val Acc=0.2186 | Time=46.4s
✅ CRNN - Test: Acc=0.2168, 

In [10]:
# === ÉTAPE 10: MODÈLES PRÉ-ENTRAÎNÉS ===

print("🎵 CHARGEMENT DES MODÈLES PRÉ-ENTRAÎNÉS")
print("=" * 60)

try:
    import torchaudio
    print(f"✅ TorchAudio version: {torchaudio.__version__}")
    
    # Dataset pour waveform (nécessaire pour Wav2Vec2, HuBERT)
    class WaveformDataset(Dataset):
        def __init__(self, df, sr=16000, max_length=16000*5):  # 5 secondes max
            self.df = df.reset_index(drop=True)
            self.sr = sr
            self.max_length = max_length
        
        def __len__(self):
            return len(self.df)
        
        def __getitem__(self, idx):
            fp = self.df.loc[idx, "file_path"]
            label = int(self.df.loc[idx, "label"])
            
            try:
                waveform, sr = torchaudio.load(fp)
                
                # Resample si nécessaire
                if sr != self.sr:
                    resampler = torchaudio.transforms.Resample(sr, self.sr)
                    waveform = resampler(waveform)
                
                # Stéréo -> Mono
                if waveform.shape[0] > 1:
                    waveform = waveform.mean(dim=0, keepdim=True)
                
                # Normalisation
                waveform = waveform / (waveform.abs().max() + 1e-8)
                
                # Padding/truncation
                if waveform.shape[1] > self.max_length:
                    waveform = waveform[:, :self.max_length]
                else:
                    pad_length = self.max_length - waveform.shape[1]
                    waveform = torch.nn.functional.pad(waveform, (0, pad_length))
                
                return waveform, torch.tensor(label).long()
                
            except Exception as e:
                print(f"❌ Erreur chargement {fp}: {e}")
                dummy_waveform = torch.zeros(1, self.max_length)
                return dummy_waveform, torch.tensor(label).long()
    
    # Création des datasets waveform
    train_wave_ds = WaveformDataset(df_train)
    val_wave_ds = WaveformDataset(df_val)
    test_wave_ds = WaveformDataset(df_test)
    
    train_wave_loader = DataLoader(train_wave_ds, batch_size=8, shuffle=True, num_workers=0)
    val_wave_loader = DataLoader(val_wave_ds, batch_size=8, shuffle=False, num_workers=0)
    test_wave_loader = DataLoader(test_wave_ds, batch_size=8, shuffle=False, num_workers=0)
    
    print("✅ Datasets waveform créés")
    
except ImportError as e:
    print(f"❌ TorchAudio non disponible: {e}")
    train_wave_loader, val_wave_loader, test_wave_loader = None, None, None

🎵 CHARGEMENT DES MODÈLES PRÉ-ENTRAÎNÉS
✅ TorchAudio version: 2.9.0
✅ Datasets waveform créés


In [11]:
# === ÉTAPE 11: ARCHITECTURES DES MODÈLES PRÉ-ENTRAÎNÉS ===

print("🧠 DÉFINITION DES MODÈLES PRÉ-ENTRAÎNÉS")
print("=" * 60)

pretrained_results = {}

# 1. Wav2Vec 2.0
if train_wave_loader is not None:
    try:
        class Wav2Vec2Classifier(nn.Module):
            def __init__(self, num_classes=NUM_CLASSES, freeze_base=True):
                super().__init__()
                bundle = torchaudio.pipelines.WAV2VEC2_BASE
                self.wav2vec2 = bundle.get_model()
                self.feature_dim = bundle._params['encoder_embed_dim']
                
                if freeze_base:
                    for param in self.wav2vec2.parameters():
                        param.requires_grad = False
                
                self.classifier = nn.Sequential(
                    nn.Linear(self.feature_dim, 256),
                    nn.ReLU(),
                    nn.Dropout(0.3),
                    nn.Linear(256, 128),
                    nn.ReLU(),
                    nn.Dropout(0.2),
                    nn.Linear(128, num_classes)
                )
            
            def forward(self, x):
                features, _ = self.wav2vec2(x)
                if len(features.shape) == 3:
                    features = features.mean(dim=1)
                return self.classifier(features)
        
        print("✅ Wav2Vec2 défini")
        
        # Entraînement Wav2Vec2
        print("\n🎵 Entraînement Wav2Vec2...")
        model_wav2vec2 = Wav2Vec2Classifier(num_classes=NUM_CLASSES)
        wav2vec2_acc, wav2vec2_f1, _ = train_model(
            model_wav2vec2, train_wave_loader, val_wave_loader, "Wav2Vec2", epochs=3, lr=1e-4
        )
        pretrained_results['Wav2Vec2'] = (wav2vec2_acc, wav2vec2_f1)
        
    except Exception as e:
        print(f"❌ Erreur Wav2Vec2: {e}")

# 2. YAMNet-like (alternative si Wav2Vec2 échoue)
class YAMNetLike(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(256, 512, kernel_size=3, padding=1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        features = self.conv_layers(x)
        features = features.view(features.size(0), -1)
        return self.classifier(features)

print("✅ YAMNet-like défini")

# Entraînement YAMNet-like
print("\n🎵 Entraînement YAMNet-like...")
model_yamnet = YAMNetLike(num_classes=NUM_CLASSES)
yamnet_acc, yamnet_f1, _ = train_model(
    model_yamnet, train_loader, val_loader, "YAMNet-like", epochs=5, lr=1e-3
)
pretrained_results['YAMNet-like'] = (yamnet_acc, yamnet_f1)

# 3. PANNs-like
class PANNsLike(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.conv_blocks = nn.Sequential(
            # Bloc 1
            nn.Conv2d(1, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            # Bloc 2
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            # Bloc 3
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
            # Bloc 4
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.conv_blocks(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

print("✅ PANNs-like défini")

# Entraînement PANNs-like
print("\n🎵 Entraînement PANNs-like...")
model_panns = PANNsLike(num_classes=NUM_CLASSES)
panns_acc, panns_f1, _ = train_model(
    model_panns, train_loader, val_loader, "PANNs-like", epochs=5, lr=1e-3
)
pretrained_results['PANNs-like'] = (panns_acc, panns_f1)

print("✅ MODÈLES PRÉ-ENTRAÎNÉS DÉFINIS ET ENTRAÎNÉS")

🧠 DÉFINITION DES MODÈLES PRÉ-ENTRAÎNÉS
✅ Wav2Vec2 défini

🎵 Entraînement Wav2Vec2...
Downloading: "https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960.pth" to /Users/lafiloche31200/.cache/torch/hub/checkpoints/wav2vec2_fairseq_base_ls960.pth


100%|██████████| 360M/360M [00:22<00:00, 17.0MB/s] 



🚀 Entraînement Wav2Vec2...
❌ Erreur chargement /Users/lafiloche31200/Projects/projet_donnee/audio/video2149.wav: TorchCodec is required for load_with_torchcodec. Please install torchcodec to use this function.
❌ Erreur chargement /Users/lafiloche31200/Projects/projet_donnee/audio/video4555.wav: TorchCodec is required for load_with_torchcodec. Please install torchcodec to use this function.
❌ Erreur chargement /Users/lafiloche31200/Projects/projet_donnee/audio/video3255.wav: TorchCodec is required for load_with_torchcodec. Please install torchcodec to use this function.
❌ Erreur chargement /Users/lafiloche31200/Projects/projet_donnee/audio/video428.wav: TorchCodec is required for load_with_torchcodec. Please install torchcodec to use this function.
❌ Erreur chargement /Users/lafiloche31200/Projects/projet_donnee/audio/video4951.wav: TorchCodec is required for load_with_torchcodec. Please install torchcodec to use this function.
❌ Erreur chargement /Users/lafiloche31200/Projects/projet_

In [1]:
# === ÉTAPE 12: VISUALISATION DES RÉSULTATS ===

print("📊 ANALYSE ET VISUALISATION DES RÉSULTATS")
print("=" * 60)

# Combinaison de tous les résultats
all_results = {**results, **pretrained_results}

# Affichage du tableau comparatif
print("\n" + "=" * 80)
print("📋 TABLEAU COMPARATIF COMPLET")
print("=" * 80)
print(f"{'Modèle':<20} {'Accuracy':<10} {'F1-Score':<10} {'Gain vs RF':<12}")
print("-" * 80)

rf_acc = results['Random Forest'][0]
for model, (acc, f1) in sorted(all_results.items(), key=lambda x: x[1][0], reverse=True):
    gain = ((acc - rf_acc) / rf_acc) * 100 if rf_acc > 0 else 0
    print(f"{model:<20} {acc:<10.4f} {f1:<10.4f} {gain:>+7.1f}%")

print("=" * 80)

# 🏆 Meilleur modèle
best_model_name = max(all_results.items(), key=lambda x: x[1][0])[0]
best_accuracy = all_results[best_model_name][0]
print(f"\n🏆 MEILLEUR MODÈLE: {best_model_name}")
print(f"🎯 Accuracy: {best_accuracy:.4f}")

# Graphique comparatif
plt.figure(figsize=(14, 8))

models = list(all_results.keys())
accuracies = [all_results[m][0] for m in models]
f1_scores = [all_results[m][1] for m in models]

# Tri par accuracy
sorted_indices = np.argsort(accuracies)[::-1]
models = [models[i] for i in sorted_indices]
accuracies = [accuracies[i] for i in sorted_indices]
f1_scores = [f1_scores[i] for i in sorted_indices]

# Graphique bars
x_pos = np.arange(len(models))
width = 0.35

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Accuracy
bars1 = ax1.bar(x_pos, accuracies, width, color='skyblue', alpha=0.8, label='Accuracy')
ax1.set_ylabel('Accuracy')
ax1.set_title('COMPARAISON DES MODÈLES - ACCURACY', fontweight='bold', fontsize=14)
ax1.set_xticks(x_pos)
ax1.set_xticklabels(models, rotation=45, ha='right')
ax1.grid(axis='y', alpha=0.3)

# Ajout des valeurs sur les bars
for bar, acc in zip(bars1, accuracies):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
             f'{acc:.4f}', ha='center', va='bottom', fontweight='bold')

# F1-Score
bars2 = ax2.bar(x_pos, f1_scores, width, color='lightcoral', alpha=0.8, label='F1-Score')
ax2.set_ylabel('F1-Score (macro)')
ax2.set_title('COMPARAISON DES MODÈLES - F1-SCORE', fontweight='bold', fontsize=14)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(models, rotation=45, ha='right')
ax2.grid(axis='y', alpha=0.3)

# Ajout des valeurs sur les bars
for bar, f1 in zip(bars2, f1_scores):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 0.01,
             f'{f1:.4f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Analyse des performances par catégorie
print("\n🔍 ANALYSE PAR CATÉGORIE DE MODÈLES")
print("=" * 50)

# Catégorisation
cnn_models = [m for m in all_results.keys() if 'CNN' in m]
pretrained_models = list(pretrained_results.keys())
baseline_models = ['Random Forest']

print(f"📊 Statistiques par catégorie:")
print(f"   • Modèles CNN: {len(cnn_models)}")
print(f"   • Modèles Pré-entraînés: {len(pretrained_models)}")
print(f"   • Modèles Baseline: {len(baseline_models)}")

# Meilleurs par catégorie
if cnn_models:
    best_cnn = max([(m, all_results[m][0]) for m in cnn_models], key=lambda x: x[1])
    print(f"   🥇 Meilleur CNN: {best_cnn[0]} ({best_cnn[1]:.4f})")

if pretrained_models:
    best_pretrained = max([(m, all_results[m][0]) for m in pretrained_models], key=lambda x: x[1])
    print(f"   🥇 Meilleur Pré-entraîné: {best_pretrained[0]} ({best_pretrained[1]:.4f})")

print(f"   📈 Amélioration vs Random Forest: {((best_accuracy - rf_acc) / rf_acc * 100):+.1f}%")

📊 ANALYSE ET VISUALISATION DES RÉSULTATS


NameError: name 'results' is not defined